[Coursework fork](https://github.com/pelinsukk/remote-sensing-lc) · See README for data and reproducibility.

# CNN classifier for landcover classification from remote sensing images

In this tasks, we will again use the same portion of the [SAT-6](https://csc.lsu.edu/~saikat/deepsat/) dataset from Louisiana State University and NASA Ames Research Center.

In this notebook, we will make use of small CNN network on the same subset of SAT-6 dataset that we used in the last exercise

In [ ]:
# CNN classifier for landcover classification
# Author: Ankit Patnala, JSC, 2025; updqated by Martin Schultz, April 2026
# Download necessary packages if right kernel is not present
# Not needed in colab
#!pip install pandas
#!pip install scikit-learn
#!pip install matplotlib

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from tqdm import tqdm

## Obtaining the data

Download the three CSVs from the [B2SHARE record](https://b2share.eudat.eu/records/2p6ra-xfj86) into `data/`. Run from the repository root. Both models use identical balanced splits.

In [ ]:
import os
from sat6 import load_course_data, channel_columns

# Default: full coursework experiment. Smaller values are useful for smoke tests.
num_train = int(os.environ.get("SAT6_TRAIN_PER_CLASS", "1000"))
num_val = int(os.environ.get("SAT6_VAL_PER_CLASS", "100"))
num_test = int(os.environ.get("SAT6_TEST_PER_CLASS", "100"))
landcover_df, labels_df, annotations, (train_idx, val_idx, test_idx) = load_course_data(
    train=num_train, validation=num_val, test=num_test
)
np.random.seed(42)

In [ ]:
landcover_df.head()

# Read annotation labels from different csv file

In [ ]:
labels = annotations[0].to_numpy()
print(labels)

**Reminder :**  The data structure that you see above is called _one-hot encoding_


In [ ]:
column_names = labels_df.columns
labels_df.head()

# Plot random samples for illustration

&#x1F479; _Task:_ Plot few samples to see visually the data and gain more insights on how to reshape the 1d data to a 4 channel image (RGB & NIR)

In [ ]:
num_classes = len(column_names)
num_samples = min(8, num_train)  # number of rows to plot
sample_idx = []
for column in column_names:
    # find all indices of a given class
    class_idx = labels_df.loc[train_idx, column] == 1
    # randomly select num_samples from this index list - make sure to avoid duplicates
    sample_idx.append(np.random.choice(class_idx.index[class_idx], size=num_samples, replace=False).tolist())

fig, ax = plt.subplots(num_samples, num_classes, figsize=(20,20))
for i in range(num_samples):
    for j in range(num_classes):
        data_index = sample_idx[j][i]
        ax[i,j].set_title(f"{labels[j]}, {data_index}")
        ax[i,j].axis('off')
        data = np.reshape(landcover_df.loc[data_index].values,(-1,28,28,4))
        ax[i,j].imshow(data[0,...,:3])

plt.show()

### Shared training, validation and test split

One seeded permutation per class assigns rows to nonoverlapping splits. The validation split is used for depth and learning-curve comparisons; the final test split is held out.

In [ ]:
assert not set(train_idx) & set(val_idx)
assert not set(train_idx) & set(test_idx)
assert not set(val_idx) & set(test_idx)
print(f"Train: {len(train_idx)}, validation: {len(val_idx)}, test: {len(test_idx)}")

In [ ]:
train_X, train_y = landcover_df.loc[train_idx], labels_df.loc[train_idx]
val_X, val_y = landcover_df.loc[val_idx], labels_df.loc[val_idx]
test_X, test_y = landcover_df.loc[test_idx], labels_df.loc[test_idx]

# Calculate Normalization Parameter using Train Dataset

In [ ]:
train_X_reshape = train_X.values.reshape((-1,28,28,4))
mean_per_channels = np.mean(train_X_reshape,axis=(0,1,2)).astype(np.float32)
std_per_channels = np.maximum(np.std(train_X_reshape, axis=(0,1,2)), 1e-6).astype(np.float32)
mean_per_channels, std_per_channels

## Define torch dataset

From here, we will make use of general workflow of PyTorch. First thing is to create a dataset. We are making use of mapped-style dataset. [Refer this page for more details](https://pytorch.org/docs/stable/data.html#map-style-datasets)

Important thing is to define the total length under function *__len__* and data corresponding to the index under *__getitem__*

In [ ]:
class SAT6Dataset(Dataset):
  def __init__(self,data_df,label_df, normalization_parameter):
    super(SAT6Dataset,self).__init__()
    self.data_df = data_df
    self.label_df = label_df
    self.normalization = normalization_parameter

  def __len__(self):
    return len(self.data_df)

  def __getitem__(self,idx):
    data = self.data_df.iloc[idx].values.reshape((28,28,4)).astype(np.float32)
    data = (data-self.normalization[0])/self.normalization[1]
    #data = data/255.0
    data = np.transpose(data,axes=(2,0,1))  # swap dimensions: channel first
    label = self.label_df.iloc[idx].values
    label_args = np.where(label==1)[0]

    return (data,label_args[0])

In [ ]:
# define two dataset one for train and other for test
sat6_train_dataset = SAT6Dataset(train_X,train_y,(mean_per_channels,std_per_channels))
sat6_test_dataset = SAT6Dataset(test_X,test_y,(mean_per_channels,std_per_channels))
sat6_val_dataset = SAT6Dataset(val_X,val_y,(mean_per_channels,std_per_channels))

Now, we will use **torch.nn.Module** to build our CNN model.

**Important points:**

*   To build modules in a list use **torch.nn.ModuleList()** instead of native list.

*   Define **forward** function to define the step-by-step mathematical transformation




In [ ]:
class CNN(torch.nn.Module):
  def __init__(self,
               input_shape,
               num_conv_layers,
               channels_each_layer,
               kernel_size,
               mlp_dimension,
               activation_func,
               pooling,
               input_channels,
               num_classes,
               norm="batch_norm"):
    super(CNN,self).__init__()
    self.num_conv_layers = num_conv_layers
    self.channels_each_layer = channels_each_layer
    assert num_conv_layers == len(channels_each_layer),"Number of conv layers does not match with given number of channels"
    self.kernel_size = ([kernel_size] * num_conv_layers
                        if type(kernel_size) == int
                        else kernel_size)
    self.mlp_dimension = mlp_dimension
    if activation_func == 'relu':
      self.activation_func = torch.nn.ReLU()
    elif activation_func == 'tanh':
      self.activation_func = torch.nn.Tanh()
    else:
      assert False, "Currently only relu and tanh are provided in this code"

    if pooling == "max":
      self.pooling = torch.nn.AvgPool2d(kernel_size=2, stride=2)
    else:
      self.pooling = torch.nn.MaxPool2d(kernel_size=2, stride=2)

    self.input_channels = input_channels
    if norm=="batch_norm":
      self.batch_norms = [torch.nn.BatchNorm2d(channels_each_layer[i])
                          for i in range(len(self.kernel_size))]
    elif norm == "identity":
      self.batch_norms = [torch.nn.Identity() for i in range(len(self.kernel_size))]
    else:
      assert False,"only batch norm is provided in this code"

    self.conv_modules = torch.nn.ModuleList()

    for i in range(num_conv_layers):
      self.conv_modules.append(
          torch.nn.Conv2d(input_channels if i==0 else channels_each_layer[i-1],
                          channels_each_layer[i],
                          self.kernel_size[i],
                          stride=1,
                          padding='same',
                          bias=True))
      self.conv_modules.append(self.batch_norms[i])
      self.conv_modules.append(self.activation_func)
      self.conv_modules.append(self.pooling)
    dim_after_final_conv_layer = (input_shape[0]//2**num_conv_layers)*(
        input_shape[1]//2**num_conv_layers)*(
            channels_each_layer[-1])
    self.mlp_layer_modules = torch.nn.ModuleList()
    self.mlp_dimension = ([mlp_dimension]
                          if type(mlp_dimension) == int
                          else mlp_dimension)

    for i in range(len(self.mlp_dimension)):
      self.mlp_layer_modules.append(
          torch.nn.Linear((dim_after_final_conv_layer
                           if i==0
                           else self.mlp_dimension[i-1]),
                          self.mlp_dimension[i]))
      # Incase one want to try batch norm uncomment below line
      #self.mlp_layer_modules.append(torch.nn.BatchNorm1d(mlp_dimension[i]))
      self.mlp_layer_modules.append(self.activation_func)
    self.mlp_layer_modules.append(
        torch.nn.Linear(self.mlp_dimension[-1],num_classes))

  def forward(self,x):
    for module in self.conv_modules:
      x = module(x)
    x = x.flatten(1)
    for module in self.mlp_layer_modules:
      x = module(x)
    return x

Define the device (**GPU** if you use GPU as your runtime) else it will run on **CPU**)
Instatiation of the CNN model is also define in this cell

In [ ]:
torch.manual_seed(42)
torch.set_num_threads(4)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
cnn = CNN((28,28),3,[32,64,128],[5,3,3],[32],"relu","max",4,6,norm="identity").to(device)

In [ ]:
# Overview of the model
print(cnn)

Define training parameters


*   Optimizer : This reulates the rate of learning. USe wither adam or SGD. For other optimizers, feel free to add them.
*   Scheduler : It updates the learning rate during the training process. We have commented here out. PLease feel free to play with it.
*   Loss function: As it is a multi-class classification problem, CrossEntropy is a suitable loss function.



In [ ]:
optim = "adam"
optimizer = torch.optim.Adam(cnn.parameters(),lr=0.001) if optim == "adam" else torch.optim.SGD(cnn.parameters(),lr=0.1)
#scheduler = torch.optim.lr_scheduler.StepLR(optimizer,step_size=10,gamma=0.1)
loss_fn = torch.nn.CrossEntropyLoss()

Previously, we have defined dataset which takes an index as a parameter and provides data corresponding to the index. In such deep learning problems, we process them in a batch. The DataLoader takes control of such requirements by defining the parameters **batch_size** and **shuffle**.

**Note:**


*   In training, we want each time to have different samples in a batch. The ***shuffle*=True** ensures that. For validation, we do not required such shuffling, hence it is set to False
*   We used the whole length of test dataset as it is small. Incase of any issue with large number of test samples, feel free to decrease the number such as $256$



In [ ]:
train_dataloader = DataLoader(sat6_train_dataset,batch_size=256,shuffle=True,drop_last=False, generator=torch.Generator().manual_seed(42))
val_dataloader = DataLoader(sat6_val_dataset,len(sat6_val_dataset), shuffle=False)

# Training Routine
Define number of epochs for the training.


In [ ]:
num_epochs = int(os.environ.get("SAT6_EPOCHS", "10"))
training_loss_per_epoch = []
validation_loss_per_epoch = []
acc_per_epoch = []

with tqdm(total=num_epochs) as pbar:
  for i in range(num_epochs):
    cnn.train()
    running_loss = 0.0
    for train_data_idx, train_data in enumerate(train_dataloader):
      optimizer.zero_grad()
      data, labels = train_data
      data = data.to(device)
      labels = labels.to(device)
      preds = cnn(data)
      train_loss = loss_fn(preds,labels)
      train_loss.backward()
      optimizer.step()
      running_loss += train_loss.item() * len(labels)
    # uncomment incase scheduler is defined above
    #scheduler.step()
    training_loss_per_epoch.append(running_loss / len(sat6_train_dataset))

    cnn.eval()
    pred_labels = []
    true_labels = []
    with torch.no_grad():
      for test_data_idx, test_data in enumerate(val_dataloader):
        data, labels = test_data
        data = data.to(device)
        labels = labels.to(device)
        preds = cnn(data)
        val_loss = loss_fn(preds,labels)
        pred_labels.extend(torch.argmax(preds,dim=1).to('cpu'))
        true_labels.extend(torch.squeeze(labels).to('cpu'))
    validation_loss_per_epoch.append(val_loss.item())
    acc = accuracy_score(true_labels,pred_labels)
    acc_per_epoch.append(acc)
    pbar.set_description(f"Epoch {i+1}/{num_epochs}: training_loss : {train_loss}, validation_loss : {val_loss} and accuracy : {acc} ")


    pbar.update(1)

The variables **training_loss_per_epoch**, **validation_loss_per_epoch**, and **acc_per_epoch** contains metrics after each epoch. Try to plot them to see how was the progress during training.

In [ ]:
# plotting loss and accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(training_loss_per_epoch)+1), training_loss_per_epoch,
             '-o', label='training loss', color='steelblue')
axes[0].plot(range(1, len(validation_loss_per_epoch)+1), validation_loss_per_epoch,
             '-o', label='validation loss', color='tomato')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss per epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, len(acc_per_epoch)+1), acc_per_epoch, '-o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation accuracy per epoch')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('final accuracy:', round(acc_per_epoch[-1], 4))

## Task 3.2

per class accuracy and example images. main difference from 2.2: cnn outputs logits so i use torch.argmax instead of np.argmax. also images need to be denormalized before displaying.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import itertools

labels_names = annotations[0].values  # 6 sinif ismi

cnn.eval()
all_preds, all_true, all_images = [], [], []

with torch.no_grad():
    for data, label in DataLoader(sat6_test_dataset, batch_size=256, shuffle=False):
        logits = cnn(data.to(device))
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_true.extend(label.cpu().numpy().astype(int))
        all_images.extend(data.cpu().numpy())

all_preds = np.array(all_preds).astype(int)
all_true  = np.array(all_true).astype(int)

print(classification_report(all_true, all_preds, labels=np.arange(6), target_names=labels_names, zero_division=0))

cm = confusion_matrix(all_true, all_preds, labels=np.arange(6))
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.set_title('confusion matrix')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(labels_names)))
ax.set_xticklabels(labels_names, rotation=45, ha='right')
ax.set_yticks(range(len(labels_names)))
ax.set_yticklabels(labels_names)
thresh = cm.max() / 2.
for i, j in itertools.product(range(len(labels_names)), range(len(labels_names))):
    ax.text(j, i, cm[i, j], ha='center', va='center',
            color='white' if cm[i, j] > thresh else 'black')
ax.set_ylabel('True')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()
score_cnn = accuracy_score(all_true, all_preds)
print(f"Held-out CNN accuracy: {score_cnn:.6f}")

In [ ]:
n = 4
correct_idx = np.where(all_preds == all_true)[0]
wrong_idx   = np.where(all_preds != all_true)[0]

def unnormalize(img):
    img = img.transpose(1, 2, 0)
    img = img * std_per_channels + mean_per_channels
    img = np.clip(img / 255.0, 0, 1)
    return img[..., :3]

fig, axes = plt.subplots(2, n, figsize=(16, 8))
fig.suptitle('Top: correct  |  Bottom: misclassified', fontsize=13)

for i, idx in enumerate(correct_idx[:n]):
    axes[0, i].imshow(unnormalize(all_images[idx]))
    axes[0, i].set_title('True: {}\nPred: {}'.format(labels[all_true[idx]], labels[all_preds[idx]]))
    axes[0, i].axis('off')

for i, idx in enumerate(wrong_idx[:n]):
    axes[1, i].imshow(unnormalize(all_images[idx]))
    axes[1, i].set_title('True: {}\nPred: {}'.format(labels[all_true[idx]], labels[all_preds[idx]]))
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## Task 3.3 - RGB only

drop NIR channel and retrain. i made a new dataset class that slices only the first 3 channels before returning.

In [ ]:
class SAT6Dataset_RGB(Dataset):
    def __init__(self, data_df, label_df, normalization_parameter):
        super().__init__()
        self.data_df   = data_df
        self.label_df  = label_df
        self.normalization = normalization_parameter

    def __len__(self):
        return len(self.data_df)

    def __getitem__(self, idx):
        data = self.data_df.iloc[idx].values.reshape((28, 28, 4)).astype(np.float32)
        data = (data - self.normalization[0]) / self.normalization[1]
        data = np.transpose(data, axes=(2, 0, 1))
        data = data[:3]  # drop NIR
        label = self.label_df.iloc[idx].values
        return (data, np.where(label == 1)[0][0])

sat6_train_rgb = SAT6Dataset_RGB(train_X, train_y, (mean_per_channels, std_per_channels))
sat6_val_rgb  = SAT6Dataset_RGB(val_X, val_y,  (mean_per_channels, std_per_channels))

train_loader_rgb = DataLoader(sat6_train_rgb, batch_size=256, shuffle=True, drop_last=False, generator=torch.Generator().manual_seed(42))
val_loader_rgb = DataLoader(sat6_val_rgb, len(sat6_val_rgb), shuffle=False)

torch.manual_seed(42)
cnn_rgb = CNN((28,28), 3, [32,64,128], [5,3,3], [32], 'relu', 'max', 3, 6, norm='identity').to(device)
optimizer_rgb = torch.optim.Adam(cnn_rgb.parameters(), lr=0.001)
loss_fn_rgb   = torch.nn.CrossEntropyLoss()

num_epochs_rgb = num_epochs
acc_rgb_per_epoch = []

print('training RGB-only CNN...')
with tqdm(total=num_epochs_rgb) as pbar:
    for epoch in range(num_epochs_rgb):
        cnn_rgb.train()
        for data, lab in train_loader_rgb:
            optimizer_rgb.zero_grad()
            loss = loss_fn_rgb(cnn_rgb(data.to(device)), lab.to(device))
            loss.backward()
            optimizer_rgb.step()
        cnn_rgb.eval()
        pl, tl = [], []
        with torch.no_grad():
            for data, lab in val_loader_rgb:
                pl.extend(torch.argmax(cnn_rgb(data.to(device)), dim=1).cpu())
                tl.extend(lab.squeeze().cpu())
        acc = accuracy_score(tl, pl)
        acc_rgb_per_epoch.append(acc)
        pbar.set_description('epoch {}  acc={:.4f}'.format(epoch+1, acc))
        pbar.update(1)

print('Validation, four bands: {:.4f}'.format(acc_per_epoch[-1]))
print('Validation, RGB:  {:.4f}'.format(acc_rgb_per_epoch[-1]))
print('drop: {:.2f} pp'.format((acc_per_epoch[-1] - acc_rgb_per_epoch[-1])*100))
sat6_test_rgb = SAT6Dataset_RGB(test_X, test_y, (mean_per_channels, std_per_channels))
with torch.no_grad():
    rgb_predictions = np.concatenate([
        cnn_rgb(x.to(device)).argmax(1).cpu().numpy()
        for x, _ in DataLoader(sat6_test_rgb, batch_size=256)
    ])
score_cnn_rgb = accuracy_score(test_y.to_numpy().argmax(1), rgb_predictions)
print(f"Held-out RGB CNN accuracy: {score_cnn_rgb:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, num_epochs+1),     acc_per_epoch,     '-o', label='4-channel', color='steelblue')
ax.plot(range(1, num_epochs_rgb+1), acc_rgb_per_epoch, '-s', label='RGB only',  color='tomato')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation accuracy')
ax.set_title('4-channel vs RGB-only')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# from the remaining channels which makes up for losing NIR a bit

## Task 3.4 — Learning rate

Compare the two fixed learning rates on validation curves. A slower learning rate may need more epochs; this experiment does not establish its eventual accuracy.

In [ ]:
torch.manual_seed(42)
train_loader_slow = DataLoader(sat6_train_dataset, batch_size=256, shuffle=True,
                               generator=torch.Generator().manual_seed(42))
cnn_slow = CNN((28,28), 3, [32,64,128], [5,3,3], [32], 'relu', 'max', 4, 6, norm='identity').to(device)
optimizer_slow = torch.optim.Adam(cnn_slow.parameters(), lr=0.0001)
loss_fn_slow   = torch.nn.CrossEntropyLoss()

acc_slow_per_epoch  = []
loss_slow_per_epoch = []

print('training with lr=0.0001...')
with tqdm(total=num_epochs) as pbar:
    for epoch in range(num_epochs):
        cnn_slow.train()
        for data, lab in train_loader_slow:
            optimizer_slow.zero_grad()
            loss = loss_fn_slow(cnn_slow(data.to(device)), lab.to(device))
            loss.backward()
            optimizer_slow.step()
        cnn_slow.eval()
        pl, tl = [], []
        with torch.no_grad():
            for data, lab in val_dataloader:
                out = cnn_slow(data.to(device))
                loss_slow_per_epoch.append(loss_fn_slow(out, lab.to(device)).item())
                pl.extend(torch.argmax(out, dim=1).cpu())
                tl.extend(lab.squeeze().cpu())
        acc = accuracy_score(tl, pl)
        acc_slow_per_epoch.append(acc)
        pbar.set_description('epoch {}  acc={:.4f}'.format(epoch+1, acc))
        pbar.update(1)

print('lr=0.001  -> {:.4f}'.format(acc_per_epoch[-1]))
print('lr=0.0001 -> {:.4f}'.format(acc_slow_per_epoch[-1]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, num_epochs+1), acc_per_epoch,      '-o', label='lr=0.001',  color='steelblue')
axes[0].plot(range(1, num_epochs+1), acc_slow_per_epoch, '-s', label='lr=0.0001', color='darkorange')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('accuracy comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, num_epochs+1), validation_loss_per_epoch, '-o', label='lr=0.001',  color='steelblue')
axes[1].plot(range(1, num_epochs+1), loss_slow_per_epoch,       '-s', label='lr=0.0001', color='darkorange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Val loss')
axes[1].set_title('validation loss comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()